# Module 1: K-Means Clustering Analysis
## Research Question: How do SLM Technologies cluster geographically and environmentally?

This notebook performs k-means clustering on WOCAT SLM Technologies data to identify geographic and environmental clusters.

## 1. Setup and Imports

In [ ]:
#!pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")

## 2. Configuration

Edit these settings to customize your analysis:

In [ ]:
# ==================== CONFIGURATION ====================

# Data file path - update this if your file is in a different location
DATA_FILE = '/Users/normankearney/Documents/CAS-AML-WOCAT/colab/datasets/wocat/wocat_technologies.csv.gz'

# Features to include in clustering (mapped to actual column names)
ESSENTIAL_FEATURES = {
    'latitude': 'numeric',
    'longitude': 'numeric',
    'altitude': 'categorical',
    'agroclimatic_zone': 'categorical',
    'slm_group_primary': 'categorical'
}

OPTIONAL_FEATURES = {
    'landuse': 'categorical',
    'degradation': 'categorical',
    'watersupply': 'categorical',
}

# K-means grid search parameters
K_MIN = 2
K_MAX = 20
RANDOM_STATE = 42
N_INIT = 10

print(f"Configuration loaded:")
print(f"  Data file: {DATA_FILE}")
print(f"  K range: {K_MIN}-{K_MAX}")

## 3. Data Loading and Preparation

In [ ]:
def load_and_prepare_data(filepath, use_optional=True):
    """
    Load WOCAT data from gzipped CSV and prepare features for clustering.
    """
    print("Loading WOCAT data from gzipped CSV...")
    
    # Load gzipped CSV
    data = pd.read_csv(filepath, compression='gzip')
    
    print(f"Original dataset shape: {data.shape}")
    
    # Select features
    features_to_use = ESSENTIAL_FEATURES.copy()
    if use_optional:
        features_to_use.update(OPTIONAL_FEATURES)
    
    # Check available columns
    available_features = {}
    for feature_name, feature_type in features_to_use.items():
        if feature_name in data.columns:
            available_features[feature_name] = feature_type
        else:
            print(f"  ⚠ Missing: {feature_name}")
    
    if not available_features:
        raise ValueError(f"No matching features found in data.")
    
    print(f"\n✓ Using {len(available_features)} features:")
    for fname in available_features.keys():
        missing = data[fname].isnull().sum()
        print(f"  - {fname}: {missing} missing values")
    
    # FILTER 1: Remove rows with invalid geographic coordinates
    data_before_geo = len(data)
    data = data[(data['latitude'] >= -90) & (data['latitude'] <= 90) &
                (data['longitude'] >= -180) & (data['longitude'] <= 180)]
    removed_geo = data_before_geo - len(data)
    if removed_geo > 0:
        print(f"\n⚠ Removed {removed_geo} rows with invalid coordinates")
    
    # FILTER 2: Remove rows with missing essential features
    essential_cols = [f for f in ESSENTIAL_FEATURES.keys() if f in available_features]
    data_before_missing = len(data)
    data = data[list(available_features.keys())].dropna(subset=essential_cols)
    removed_missing = data_before_missing - len(data)
    if removed_missing > 0:
        print(f"Removed {removed_missing} rows with missing essential features")
    
    print(f"\nAfter filtering: {data.shape[0]} technologies ready for clustering")
    print(f"Total removed: {removed_geo + removed_missing} rows\n")
    
    # Prepare feature matrix
    X = prepare_features(data, available_features)
    
    return X, list(available_features.keys()), data

def prepare_features(data, features_dict):
    """
    Prepare and standardize features for clustering.
    
    Categorical variables are one-hot encoded.
    Numeric variables are standardized (mean=0, std=1).
    """
    print("Preparing features...")
    feature_arrays = []
    
    for feature_name, feature_type in features_dict.items():
        if feature_type == 'numeric':
            # Standardize numeric features
            values = data[feature_name].values.reshape(-1, 1)
            scaler = StandardScaler()
            scaled = scaler.fit_transform(values)
            feature_arrays.append(scaled)
            print(f"  ✓ {feature_name}: numeric (scaled)")
            print(f"      Range: [{data[feature_name].min():.2f}, {data[feature_name].max():.2f}]")
            
        elif feature_type == 'categorical':
            # One-hot encode categorical features
            n_unique = data[feature_name].nunique()
            
            if n_unique > 50:
                # Keep top 20 categories, group rest as "other"
                top_categories = data[feature_name].value_counts().head(20).index
                data_encoded = data[feature_name].copy()
                data_encoded = data_encoded.where(data_encoded.isin(top_categories), "other")
            else:
                data_encoded = data[feature_name]
            
            encoded = pd.get_dummies(data_encoded, 
                                   prefix=feature_name, 
                                   drop_first=False,
                                   dummy_na=True)
            feature_arrays.append(encoded.values)
            print(f"  ✓ {feature_name}: categorical ({len(encoded.columns)} categories)")
    
    X = np.hstack(feature_arrays)
    print(f"\nFinal feature matrix shape: {X.shape}")
    print(f"  - {X.shape[0]} samples (SLM technologies)")
    print(f"  - {X.shape[1]} features (after one-hot encoding)\n")
    
    return X

In [ ]:
# Load data
X, feature_names, data = load_and_prepare_data(DATA_FILE, use_optional=True)


## 3.5 Data Quality Check

Before clustering, we need to ensure our geographic coordinates are valid. Invalid coordinates can skew results and create spurious clusters.

In [ ]:
# ==================== DATA QUALITY CHECK ====================

# Load the raw data first to check quality (before feature preparation)
print("\n" + "="*80)
print("DATA QUALITY CHECK")
print("="*80 + "\n")

raw_data = pd.read_csv(DATA_FILE, compression='gzip')

print(f"Raw dataset shape: {raw_data.shape}\n")

# Check 1: Valid geographic coordinates
print("CHECK 1: Valid Geographic Coordinates")
print("-" * 80)

# Longitude should be -180 to 180
invalid_lon = raw_data[(raw_data['longitude'] < -180) | (raw_data['longitude'] > 180)]
print(f"Invalid longitude values (outside [-180, 180]): {len(invalid_lon)}")

if len(invalid_lon) > 0:
    print("\nRows with invalid longitude:")
    cols_to_show = ['technology_id', 'name', 'country_name', 'latitude', 'longitude']
    cols_to_show = [c for c in cols_to_show if c in invalid_lon.columns]
    print(invalid_lon[cols_to_show].to_string())
    print()

# Latitude should be -90 to 90
invalid_lat = raw_data[(raw_data['latitude'] < -90) | (raw_data['latitude'] > 90)]
print(f"Invalid latitude values (outside [-90, 90]): {len(invalid_lat)}")

if len(invalid_lat) > 0:
    print("\nRows with invalid latitude:")
    cols_to_show = ['technology_id', 'name', 'country_name', 'latitude', 'longitude']
    cols_to_show = [c for c in cols_to_show if c in invalid_lat.columns]
    print(invalid_lat[cols_to_show].to_string())
    print()

# Check 2: Summary statistics for coordinates
print("\nCHECK 2: Coordinate Statistics")
print("-" * 80)
valid_coords = raw_data[(raw_data['longitude'] >= -180) & (raw_data['longitude'] <= 180) &
                        (raw_data['latitude'] >= -90) & (raw_data['latitude'] <= 90)]
print(f"Rows with valid coordinates: {len(valid_coords)}\n")

print("Latitude statistics:")
print(f"  Min: {valid_coords['latitude'].min():.4f}°")
print(f"  Max: {valid_coords['latitude'].max():.4f}°")
print(f"  Mean: {valid_coords['latitude'].mean():.4f}°")
print(f"  Std: {valid_coords['latitude'].std():.4f}°\n")

print("Longitude statistics:")
print(f"  Min: {valid_coords['longitude'].min():.4f}°")
print(f"  Max: {valid_coords['longitude'].max():.4f}°")
print(f"  Mean: {valid_coords['longitude'].mean():.4f}°")
print(f"  Std: {valid_coords['longitude'].std():.4f}°\n")

# Check 3: Missing values in essential columns
print("CHECK 3: Missing Values in Essential Features")
print("-" * 80)
for feat in ESSENTIAL_FEATURES.keys():
    if feat in raw_data.columns:
        missing = raw_data[feat].isnull().sum()
        pct = (missing / len(raw_data)) * 100
        print(f"{feat:20} {missing:4d} missing ({pct:5.1f}%)")

print("\n" + "="*80)
print(f"\nRecommendation: Remove {len(invalid_lon) + len(invalid_lat)} rows with invalid coordinates")
print("This is handled automatically in the clustering process.\n")
print("="*80 + "\n")


## 4. K-Means Grid Search

In [ ]:
def grid_search_kmeans(X, k_min=K_MIN, k_max=K_MAX, random_state=RANDOM_STATE):
    """
    Perform grid search to find optimal number of clusters.
    """
    print(f"Performing grid search for k={k_min} to k={k_max}...\n")
    
    results = []
    models = {}
    
    for k in range(k_min, k_max + 1):
        print(f"  k={k:2d}: ", end='', flush=True)
        
        kmeans = KMeans(n_clusters=k, 
                       random_state=random_state,
                       n_init=N_INIT)
        labels = kmeans.fit_predict(X)
        
        # Calculate metrics
        silhouette = silhouette_score(X, labels)
        davies_bouldin = davies_bouldin_score(X, labels)
        calinski_harabasz = calinski_harabasz_score(X, labels)
        inertia = kmeans.inertia_
        
        results.append({
            'k': k,
            'silhouette': silhouette,
            'davies_bouldin': davies_bouldin,
            'calinski_harabasz': calinski_harabasz,
            'inertia': inertia
        })
        
        models[k] = kmeans
        
        print(f"Silhouette={silhouette:6.3f}  DB={davies_bouldin:6.3f}  CH={calinski_harabasz:8.1f}")
    
    results_df = pd.DataFrame(results)
    print(f"\nGrid search complete!\n")
    
    return results_df, models


# Run grid search
results_df, models = grid_search_kmeans(X)

## 5. Evaluation Metrics

In [ ]:
# Display evaluation metrics table
print("EVALUATION METRICS:\n")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(results_df.to_string(index=False))
print()

In [ ]:
def find_optimal_k(results_df):
    """
    Identify optimal number of clusters based on multiple metrics.
    """
    optimal = {
        'silhouette': int(results_df.loc[results_df['silhouette'].idxmax(), 'k']),
        'davies_bouldin': int(results_df.loc[results_df['davies_bouldin'].idxmin(), 'k']),
        'calinski_harabasz': int(results_df.loc[results_df['calinski_harabasz'].idxmax(), 'k']),
    }
    
    print("="*80)
    print("OPTIMAL NUMBER OF CLUSTERS (by metric):")
    print("="*80)
    print(f"  Silhouette Score:      k={optimal['silhouette']}")
    print(f"  Davies-Bouldin Index:  k={optimal['davies_bouldin']}")
    print(f"  Calinski-Harabasz:     k={optimal['calinski_harabasz']}")
    print("\nRECOMMENDATION: Review visualizations and select k based on")
    print("interpretability and domain knowledge about SLM Technologies.\n")
    print("="*80 + "\n")
    
    return optimal


optimal = find_optimal_k(results_df)

## 6. Visualization: Evaluation Metrics

In [ ]:
def plot_evaluation_metrics(results_df):
    """
    Plot k-means evaluation metrics to identify elbow point.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('K-Means Clustering Evaluation Metrics', fontsize=16, fontweight='bold')
    
    # Silhouette Score (higher is better)
    axes[0, 0].plot(results_df['k'], results_df['silhouette'], 'o-', linewidth=2.5, markersize=8, color='#2E86AB')
    axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=11)
    axes[0, 0].set_ylabel('Silhouette Score', fontsize=11)
    axes[0, 0].set_title('Silhouette Score (higher is better)', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_xticks(results_df['k'])
    
    # Davies-Bouldin Index (lower is better)
    axes[0, 1].plot(results_df['k'], results_df['davies_bouldin'], 'o-', linewidth=2.5, markersize=8, color='#A23B72')
    axes[0, 1].set_xlabel('Number of Clusters (k)', fontsize=11)
    axes[0, 1].set_ylabel('Davies-Bouldin Index', fontsize=11)
    axes[0, 1].set_title('Davies-Bouldin Index (lower is better)', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_xticks(results_df['k'])
    
    # Calinski-Harabasz Score (higher is better)
    axes[1, 0].plot(results_df['k'], results_df['calinski_harabasz'], 'o-', linewidth=2.5, markersize=8, color='#F18F01')
    axes[1, 0].set_xlabel('Number of Clusters (k)', fontsize=11)
    axes[1, 0].set_ylabel('Calinski-Harabasz Score', fontsize=11)
    axes[1, 0].set_title('Calinski-Harabasz Score (higher is better)', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_xticks(results_df['k'])
    
    # Inertia (Elbow method - lower is better)
    axes[1, 1].plot(results_df['k'], results_df['inertia'], 'o-', linewidth=2.5, markersize=8, color='#C73E1D')
    axes[1, 1].set_xlabel('Number of Clusters (k)', fontsize=11)
    axes[1, 1].set_ylabel('Within-Cluster Sum of Squares', fontsize=11)
    axes[1, 1].set_title('Elbow Plot (lower is better)', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_xticks(results_df['k'])
    
    plt.tight_layout()
    plt.savefig('kmeans_evaluation_metrics.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: kmeans_evaluation_metrics.png")


plot_evaluation_metrics(results_df)

## 7. Run Final Analysis with Selected K

**IMPORTANT:** Review the evaluation metrics above. If you want to use a different k value, edit `SELECTED_K` in the Configuration cell (section 2) and re-run from there.

In [ ]:
# Which k to use for final analysis (change this after reviewing evaluation metrics)
SELECTED_K = 6
print(f"SELECTED k = {SELECTED_K} for final analysis\n")

# Get final model and labels
final_model = models[SELECTED_K]
final_labels = final_model.labels_

# Print cluster sizes
unique, counts = np.unique(final_labels, return_counts=True)
print(f"Cluster sizes for k={SELECTED_K}:")
for cluster_id, count in zip(unique, counts):
    pct = (count / len(final_labels)) * 100
    print(f"  Cluster {cluster_id}: {count:4d} technologies ({pct:5.1f}%)")
print()

## 8. Visualization: Geographic Clusters

In [ ]:
def plot_geographic_clusters(data, labels, k):
    """
    Plot clusters on geographic map (latitude vs longitude).
    """
    plt.figure(figsize=(14, 10))
    
    # Use a colormap with enough distinct colors
    if k <= 10:
        cmap = 'tab10'
    elif k <= 20:
        cmap = 'tab20'
    else:
        cmap = 'hsv'
    
    scatter = plt.scatter(data['longitude'], data['latitude'], 
                         c=labels, cmap=cmap, s=100, alpha=0.6, 
                         edgecolors='black', linewidth=0.5)
    
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.title(f'Geographic Distribution of SLM Technology Clusters (k={k})', 
              fontsize=14, fontweight='bold')
    plt.colorbar(scatter, label='Cluster ID')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'geographic_clusters_k{k}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: geographic_clusters_k{k}.png")


plot_geographic_clusters(data, final_labels, SELECTED_K)

## 9. Visualization: Cluster Characteristics

In [ ]:
def plot_cluster_characteristics(data, labels, k):
    """
    Visualize key characteristics of each cluster.
    """
    data_with_clusters = data.copy()
    data_with_clusters['cluster'] = labels
    
    # Get categorical columns to visualize
    categorical_cols = []
    for col in ['agroclimatic_zone', 'altitude', 'slm_group_primary', 
                'landuse_primary', 'degradation_primary', 'watersupply']:
        if col in data.columns:
            categorical_cols.append(col)
    
    if len(categorical_cols) == 0:
        print("⚠ No categorical columns found for cluster characteristics plot")
        return
    
    n_cols = min(len(categorical_cols), 3)
    n_rows = (len(categorical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4.5*n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1 or n_cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    for idx, col in enumerate(categorical_cols):
        if col in data_with_clusters.columns:
            cluster_distribution = pd.crosstab(data_with_clusters['cluster'], 
                                              data_with_clusters[col], 
                                              normalize='index')
            
            # Limit to top categories if too many
            if cluster_distribution.shape[1] > 10:
                cluster_distribution = cluster_distribution.iloc[:, :10]
            
            cluster_distribution.plot(kind='bar', ax=axes[idx], width=0.8)
            axes[idx].set_title(f'{col}', fontweight='bold', fontsize=11)
            axes[idx].set_xlabel('Cluster')
            axes[idx].set_ylabel('Proportion')
            axes[idx].legend(title=col, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            axes[idx].tick_params(axis='x', rotation=0)
            axes[idx].grid(axis='y', alpha=0.3)
    
    # Remove empty subplots
    for idx in range(len(categorical_cols), len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.savefig(f'cluster_characteristics_k{k}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: cluster_characteristics_k{k}.png")


plot_cluster_characteristics(data, final_labels, SELECTED_K)

## 10. Cluster Interpretation

In [ ]:
def interpret_clusters(data, labels, k, feature_names):
    """
    Generate interpretable cluster summaries.
    """
    data_with_clusters = data.copy()
    data_with_clusters['cluster'] = labels
    
    print("\n" + "="*80)
    print(f"CLUSTER INTERPRETATION (k={k})")
    print("="*80)
    
    for cluster_id in range(k):
        cluster_data = data_with_clusters[data_with_clusters['cluster'] == cluster_id]
        n_samples = len(cluster_data)
        pct = (n_samples / len(data_with_clusters)) * 100
        
        print(f"\n{'─'*80}")
        print(f"CLUSTER {cluster_id:2d}: {n_samples:4d} technologies ({pct:5.1f}%)")
        print(f"{'─'*80}")
        
        # Numeric features: show mean values
        numeric_cols = cluster_data.select_dtypes(include=[np.number]).columns.tolist()
        if 'cluster' in numeric_cols:
            numeric_cols.remove('cluster')
        
        if numeric_cols:
            print("\nNumeric Features:")
            for col in numeric_cols[:5]:  # Limit to top 5
                if col in ['latitude', 'longitude']:
                    mean_val = cluster_data[col].mean()
                    std_val = cluster_data[col].std()
                    print(f"  {col:25} Mean={mean_val:8.2f}, Std={std_val:6.2f}")
        
        # Categorical features: show distribution
        categorical_cols = ['agroclimatic_zone', 'slm_group_primary', 'landuse_primary', 
                           'degradation_primary', 'watersupply']
        
        for col in categorical_cols:
            if col in cluster_data.columns:
                print(f"\n{col}:")
                value_counts = cluster_data[col].value_counts().head(5)
                for value, count in value_counts.items():
                    pct_cat = (count / n_samples) * 100
                    if pd.isna(value):
                        value_str = "(Missing)"
                    else:
                        value_str = str(value)[:40]
                    print(f"  - {value_str:40} {count:3d} ({pct_cat:5.1f}%)")


interpret_clusters(data, final_labels, SELECTED_K, feature_names)

## 11. Save Results

In [ ]:
# Save grid search results
results_df.to_csv('kmeans_grid_search_results.csv', index=False)
print(f"✓ Saved: kmeans_grid_search_results.csv")

# Save data with cluster assignments
output_data = data.copy()
output_data['cluster'] = final_labels
output_data.to_csv(f'wocat_data_with_clusters_k{SELECTED_K}.csv', index=False)
print(f"✓ Saved: wocat_data_with_clusters_k{SELECTED_K}.csv")

print(f"\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)

## Summary

### Next Steps

1. **Try different k values:** Edit `SELECTED_K` in the Configuration cell and re-run the analysis
2. **Investigate cluster patterns:** Use the cluster interpretations and visualizations to understand what makes each cluster distinct
3. **Domain validation:** Compare the clusters with known geographic and agronomic regions
4. **Move to Module 2:** Use the clustered data for supervised learning analysis of biodiversity improvements

### Output Files

- `kmeans_evaluation_metrics.png` - Comparison of 4 evaluation metrics
- `geographic_clusters_k{k}.png` - Map showing geographic distribution of clusters
- `cluster_characteristics_k{k}.png` - Feature distributions by cluster
- `kmeans_grid_search_results.csv` - Raw evaluation metrics for k=2 to k=20
- `wocat_data_with_clusters_k{k}.csv` - Original data with cluster assignments

# Understanding the Clustering Metrics

This cell explains the four metrics used to evaluate k-means clustering quality.

---

## 1. Silhouette Score (range: -1 to 1)

**What it measures:** How well each point fits in its cluster compared to other clusters.

**How it works:**
- For each point, calculate:
  - `a` = average distance to other points in the same cluster (cohesion—lower is better)
  - `b` = average distance to points in the nearest other cluster (separation—higher is better)
  - Silhouette = (b - a) / max(a, b)

**Interpretation:**
- **+1** = Point is very close to its cluster, far from others (perfect)
- **0** = Point is on the boundary between clusters
- **-1** = Point is closer to other clusters than its own (worst case)

**Rule of thumb:**
- **> 0.7** = Strong structure detected
- **0.5-0.7** = Reasonable structure
- **0.25-0.5** = Weak structure
- **< 0.25** = No substantial structure

---

## 2. Davies-Bouldin Index (lower is better, no fixed range)

**What it measures:** The average similarity between each cluster and its most similar cluster.

**How it works:**
- For each cluster, find which other cluster is most similar to it
- Calculate how similar they are (ratio of within-cluster variance)
- Average this across all clusters
- Lower values mean clusters are more distinct from each other

**Interpretation:**
- **< 1.0** = Excellent cluster separation
- **1.0-1.5** = Good separation
- **1.5-2.5** = Reasonable separation
- **> 2.5** = Poor separation, clusters overlap significantly

**Why it matters:** Davies-Bouldin directly measures whether your clusters are actually different from each other, not just whether points cluster tightly.

---

## 3. Calinski-Harabasz Index (higher is better)

**What it measures:** The ratio of between-cluster variance to within-cluster variance.

**How it works:**
- **Between-cluster variance:** How far apart cluster centers are from each other
- **Within-cluster variance:** How spread out points are within each cluster
- Index = (between-cluster / within-cluster) × scaling factor

A high ratio means clusters are compact and well-separated.

**Interpretation:**
- **> 100** = Excellent structure
- **50-100** = Good structure
- **25-50** = Acceptable structure
- **< 25** = Poor structure

**Why it matters:** Unlike silhouette, this metric rewards both tightness AND separation, making it good for finding the "natural" number of clusters.

---

## 4. Inertia (Elbow Method, lower is better)

**What it measures:** Sum of squared distances from each point to its cluster center.

**How it works:**
- For each point: calculate distance to its cluster center, square it
- Sum across all points
- This is what k-means algorithm actually minimizes

**Interpretation:**
- Always decreases as k increases (adding more clusters means lower total distance)
- Look for the "elbow"—where the curve changes slope dramatically
- The elbow suggests the point of diminishing returns

**Why it matters:** Inertia directly reflects what k-means is optimizing for. The elbow is where adding more clusters stops being "worth it."

---

## Comparing the Metrics

| Metric | What it Favors | Interpretation |
|--------|---|---|
| **Silhouette** | Well-separated, cohesive clusters | How much individual points fit |
| **Davies-Bouldin** | Distinct, non-overlapping clusters | Whether clusters are actually different |
| **Calinski-Harabasz** | Compact clusters with distance between centers | Balance of tightness + separation |
| **Inertia (Elbow)** | Lowest total within-cluster distance | Diminishing returns |

---

## How to Use These Results

### Step 1: Look at the Evaluation Metrics Table
Review the numbers for each k value to spot patterns.

### Step 2: Review the Evaluation Metrics Plot
Four subplots show the trends:
- **Silhouette**: Should peak at an optimal k
- **Davies-Bouldin**: Should have a minimum at an optimal k
- **Calinski-Harabasz**: Should peak, then decline
- **Elbow**: Look for a bend in the curve

### Step 3: Look for Consensus
- **If metrics agree** (e.g., all favor k=3): That's likely your optimal k
- **If metrics disagree**: Consider your research goals
  - Need tight, distinct clusters? → favor silhouette/davies-bouldin
  - Need balance? → favor calinski-harabasz
  - Want granular sub-groups? → use higher k even if metrics favor lower

### Step 4: Validate with Domain Knowledge
- Do the resulting clusters make geographic/agronomic sense?
- Can you interpret and explain each cluster?
- Are cluster sizes reasonable (not one huge cluster + tiny ones)?

---

## Common Pitfalls to Avoid

❌ **Don't automatically use the k with the highest silhouette score** if it doesn't make domain sense

❌ **Don't ignore all metrics in favor of one** — look for consensus

❌ **Don't use k=1** — that's "all data is one cluster" (trivially high silhouette)

❌ **Don't use very high k** — you'll get arbitrarily small clusters that don't generalize

✅ **Do consider interpretability** — can you explain each cluster to a colleague?

✅ **Do try multiple k values** — see how geography and technology patterns change

✅ **Do validate results** — do clusters align with known regions or agronomic zones?